# 7장 (실습 2): [토이 프로젝트] 내 PC에 미니 제로트러스트 랩 만들기

> 책 『AI 보안, 클라우드에서 시작한다』 7장 [실습] 2번을 노트북에서 단계별로 실행하는 버전입니다. 그림 7-2의 네 계층을 오픈소스와 짧은 Python 코드로 연결해, 내 PC 한 대 안에 작은 "회사"를 만듭니다. 소요 시간은 1~2시간입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. 요청마다 신원·역할·기기·위험 점수를 보고 **판단(②)하고 집행(③)하는** 흐름을 직접 확인합니다.
2. PEP를 거치지 않는 **우회 경로가 없어야** 하는 이유와, PEP 자체가 침해되면 무엇이 열리는지 확인합니다.
3. ④ 위험 탐지 결과가 몇 초 만에 **다음 판단을 바꾸는** 순간을 보고, Trace ID로 한 요청을 끝까지 따라갑니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 랩 구성 | 예정 |
| 1 | 실습 파일 준비와 실행 | 예정 |
| 2 | 정책 엔진의 규칙 읽기 | 예정 |
| 3 | 신원과 역할에 따라 달라지는 결과 | 예정 |
| 4 | 신원은 정상인데 차단되는 경우 | 예정 |
| 5 | 우회해 보기 | 예정 |
| 6 | 위험 탐지가 정책을 바꾸는 순간 | 예정 |
| 7 | Trace ID로 한 요청 따라가기 | 예정 |
| 8 | 체크포인트와 과제 | 예정 |
| 9 | 정리 | 예정 |

## 실습 시나리오

| 그림 7-2의 계층 | 랩의 구성요소 | 하는 일 |
|---|---|---|
| ① 사용자 | `try.sh`(요청 보내기), `devices.json`(등록 기기 목록) | 사용자·기기·접속 국가를 바꿔 가며 요청 |
| ② 인증·권한관리 | Keycloak 26(IdP), OPA 1.0(정책 엔진) | 로그인·토큰 발급, 요청마다 허용·추가 인증·차단 판단 |
| ③ 컨트롤 | `pep.py`(접근 게이트웨이) | 토큰 검증, OPA에 판단 요청, 결정 집행, 접근 로그 기록 |
| ④ 위험 탐지 | `detector.py` | 접근 로그를 3초마다 분석해 사용자별 위험 점수를 OPA에 반영 |
| 보호 대상 | `erp`, `aichat` | ERP(온프레미스 업무시스템)와 사내 AI 챗봇 역할의 작은 앱 |

```text
[try.sh] ──▶ [PEP :8000] ──판단 요청──▶ [OPA :8181] ◀──위험 점수── [detector]
   │             │                                                   ▲
   └─로그인─▶ [Keycloak :8080]   └──허용된 요청만──▶ [erp] [aichat]   │
                 │                     (내부망 back, 외부 포트 없음)    │
                 └──────────────── logs/access.jsonl ─────────────────┘
```

보호 대상 앱은 외부와 단절된 내부망(`back`)에만 붙어 있고 포트를 열지 않습니다. PEP를 거치지 않고는 닿을 길이 없습니다. 본문의 "우회 경로 없음" 요건을 구성으로 표현한 것입니다.

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Docker Compose가 필요합니다. 처음에는 Keycloak·OPA 이미지 내려받기와 서비스 이미지 빌드에 몇 분 걸립니다. 8000, 8080, 8181 포트가 비어 있어야 합니다.
>
> **2. 노트북 열기**<br>
> 처음이라면 [SETUP.md](../SETUP.md)의 6단계(VS Code와 Jupyter 노트북 연결)를 먼저 한 번 해 둡니다. 그다음에는 아래 두 방법 가운데 하나로 이 노트북을 엽니다.
> - **VS Code**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs && code .`를 실행합니다. 창 왼쪽 아래에 `WSL: Ubuntu-24.04`가 보이면, 왼쪽 탐색기에서 `ch07-zero-trust-lab` 폴더를 펼쳐 이 노트북을 엽니다. 그리고 오른쪽 위 커널 표시를 눌러 **Select Another Kernel...** → **Python Environments...** → `.venv-lab`을 고릅니다. 목록에 `.venv-lab`이 없으면 6단계 ⑥의 2번처럼 `Python: Select Interpreter` → **Enter interpreter path...**로 `~/.venv-lab/bin/python`의 위치를 먼저 알려 줍니다.
> - **웹 브라우저(Jupyter Lab)**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs/ch07-zero-trust-lab && ~/.venv-lab/bin/jupyter lab`을 실행하고, 화면에 나온 `http://localhost:8888/...` 주소를 Windows의 브라우저에서 엽니다.
> - 노트북 파일만 따로 내려받거나 다른 폴더로 옮기지 마세요. 노트북은 자기가 들어 있는 `ch07-zero-trust-lab` 폴더의 실습 파일을 읽습니다. 위 두 방법처럼 저장소 안에서 열면 따로 신경 쓸 것은 없습니다.
>
> **3. 기준 버전**<br>
> - 이 노트북은 2026년 10월 23일 Ubuntu 24.04(WSL2), Docker 29.1.3, Keycloak 26.0.7, OPA 1.0.0, Python 3.12에서 끝까지 실행해 확인했습니다. 실행 결과가 저장된 버전은 `ch07_lab2_mini_zero_trust_executed.ipynb`입니다.

> ⚠️ **주의**
> 이 랩의 계정(`kim`, `lee`, 비밀번호 `Passw0rd!`)과 Keycloak 관리자 계정(`admin`/`admin`)은 실습용입니다. `try.sh`가 쓰는 Password Grant 로그인 방식은 OAuth 2.1에서 빠진 방식으로, 실제 시스템에서는 쓰지 않습니다. 실무에서는 Authorization Code + PKCE와 피싱 저항 MFA(7-1)를 씁니다.

## 0. 실습 준비와 랩 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 랩 구성 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 실습 파일 준비와 실행 | 예정 |
| 2 | 정책 엔진의 규칙 읽기 | 예정 |
| 3 | 신원과 역할에 따라 달라지는 결과 | 예정 |
| 4 | 신원은 정상인데 차단되는 경우 | 예정 |
| 5 | 우회해 보기 | 예정 |
| 6 | 위험 탐지가 정책을 바꾸는 순간 | 예정 |
| 7 | Trace ID로 한 요청 따라가기 | 예정 |
| 8 | 체크포인트와 과제 | 예정 |
| 9 | 정리 | 예정 |

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "docker": "docker --version",
    "docker compose": "docker compose version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

### 0.1. 요청 보내기 함수

`try.sh`는 ① Keycloak에서 토큰을 받고 ② 토큰과 기기 ID·접속 국가를 붙여 PEP에 요청합니다. 노트북에서는 같은 일을 하는 파이썬 함수 `try_()`를 만들어, 결과를 표로 모아 보기 쉽게 합니다.

In [ ]:
print((LAB / "try.sh").read_text())

In [ ]:
import urllib.error
import urllib.parse
import urllib.request

WORK = Path.home() / "zt-lab"


def token(user):
    """Keycloak에서 토큰을 받습니다(실습용 Password Grant)."""
    form = urllib.parse.urlencode({"client_id": "zt-lab", "grant_type": "password",
                                   "username": user, "password": "Passw0rd!"}).encode()
    with urllib.request.urlopen("http://localhost:8080/realms/zt/protocol/openid-connect/token", form) as r:
        return json.load(r)["access_token"]


def try_(user, device, app, country="KR", method="GET"):
    """try.sh와 같은 요청을 보내고 PEP의 응답(JSON)을 돌려줍니다."""
    req = urllib.request.Request(f"http://localhost:8000/{app}/home", method=method, headers={
        "Authorization": f"Bearer {token(user)}", "X-Device-Id": device, "X-Country": country})
    try:
        with urllib.request.urlopen(req) as r:
            return json.load(r)
    except urllib.error.HTTPError as e:          # 401(추가 인증)·403(차단)도 JSON 본문을 읽는다
        return json.load(e)


def show(requests):
    """여러 요청을 보내고 결과를 표로 보여 줍니다."""
    rows = []
    for args in requests:
        r = try_(*args)
        full = (*args, *("KR", "GET")[len(args) - 3:])     # 생략한 국가·메서드는 기본값으로 표시
        rows.append((*full, f"**{r['decision']}**",
                     ", ".join(r.get("reasons") or []) or "-", r["trace"]))
    table(["사용자", "기기", "앱", "국가", "메서드", "결정", "사유", "Trace"], rows)

## 1. 실습 파일 준비와 실행

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 랩 구성 | ✅ 완료 |
| 1 | 실습 파일 준비와 실행 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 정책 엔진의 규칙 읽기 | 예정 |
| 3 | 신원과 역할에 따라 달라지는 결과 | 예정 |
| 4 | 신원은 정상인데 차단되는 경우 | 예정 |
| 5 | 우회해 보기 | 예정 |
| 6 | 위험 탐지가 정책을 바꾸는 순간 | 예정 |
| 7 | Trace ID로 한 요청 따라가기 | 예정 |
| 8 | 체크포인트와 과제 | 예정 |
| 9 | 정리 | 예정 |

이 폴더를 `~/zt-lab`으로 복사해 그 안에서 실행합니다. 4단계와 과제에서 `devices.json`과 정책 파일을 고치는데, 저장소의 원본은 그대로 두기 위해서입니다.

> 📌 (참고) 노트북을 다시 실행할 때를 위해, 같은 랩이 떠 있으면 먼저 내립니다(`down -v`). 처음에는 이미지 내려받기와 빌드에 몇 분 걸립니다.

In [ ]:
%%bash
[ -d ~/zt-lab ] && (cd ~/zt-lab && docker compose down -v -t 1 >/dev/null 2>&1)
rm -rf ~/zt-lab && cp -r . ~/zt-lab && cd ~/zt-lab
rm -f logs/access.jsonl *.ipynb
docker compose up -d --build 2>&1 | grep -E 'Started|Built|Error' | sort -u
docker compose ps --format 'table {{.Service}}\t{{.State}}'

Keycloak은 뜨는 데 30초쯤 걸립니다. `http://localhost:8080/realms/zt`가 JSON을 돌려주면 준비된 것입니다.

In [ ]:
%%bash
until curl -sf localhost:8080/realms/zt >/dev/null; do sleep 2; done
curl -s localhost:8080/realms/zt | python3 -c 'import sys,json; d=json.load(sys.stdin); print("realm:", d["realm"], "| 토큰 발급 주소:", d["token-service"])'

> 실습 계정: `kim`(임직원, 역할 employee), `lee`(협력사 직원, 역할 partner). 비밀번호는 둘 다 `Passw0rd!`입니다.

토큰 안에 무엇이 들어 있는지 열어 봅니다. 토큰(JWT)은 점(`.`)으로 나뉜 세 부분 중 가운데가 내용(claims)입니다. 서명 검증 없이 내용만 읽는 것은 누구나 할 수 있으므로, **토큰에 비밀을 넣으면 안 됩니다**.

In [ ]:
import base64

payload = token("kim").split(".")[1]
claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
print({k: claims[k] for k in ["iss", "preferred_username", "exp"]})
print("역할:", claims["realm_access"]["roles"])

## 2. 정책 엔진의 규칙 읽기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 랩 구성 | ✅ 완료 |
| 1 | 실습 파일 준비와 실행 | ✅ 완료 |
| 2 | 정책 엔진의 규칙 읽기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | 신원과 역할에 따라 달라지는 결과 | 예정 |
| 4 | 신원은 정상인데 차단되는 경우 | 예정 |
| 5 | 우회해 보기 | 예정 |
| 6 | 위험 탐지가 정책을 바꾸는 순간 | 예정 |
| 7 | Trace ID로 한 요청 따라가기 | 예정 |
| 8 | 체크포인트와 과제 | 예정 |
| 9 | 정리 | 예정 |

`policy/zt.rego`가 ② 정책 엔진의 판단 규칙입니다. 다음 중 하나라도 해당하면 차단합니다.

- 등록되지 않은 기기, 기기 소유자와 사용자 불일치, EDR이 동작하지 않는 기기
- 역할에 허용되지 않은 앱 (ERP는 임직원만, AI 챗봇은 임직원과 협력사)
- 보안 패치가 적용되지 않은 기기에서의 쓰기(POST) 요청
- 위험 점수 70 이상 (40~69는 차단 대신 추가 인증 요구)

In [ ]:
print((WORK / "policy" / "zt.rego").read_text())

### 2.1. 정책 테스트

규칙이 의도대로 동작하는지 정책 테스트부터 돌려 봅니다. 정책도 코드처럼 테스트를 거쳐 배포해야 합니다.

In [ ]:
%%bash
cd ~/zt-lab && docker compose exec -T opa opa test /policy -v

### 2.2. 정책에 직접 물어보기

OPA는 HTTP API로 "이 입력이면 결정이 무엇인가"를 바로 답합니다. PEP가 하는 일이 바로 이 질문입니다. 사용자·역할·기기·메서드를 바꿔 가며 물어 정책을 표로 펼쳐 봅니다. 토큰도 앱도 필요 없이 **정책만 따로 시험**할 수 있다는 점이 판단(②)과 집행(③)을 나눈 이점입니다.

In [ ]:
def opa_decide(**inp):
    req = urllib.request.Request("http://localhost:8181/v1/data/zt/decision",
                                 json.dumps({"input": inp}).encode(), {"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as r:
        d = json.load(r)["result"]
    kind = "ALLOW" if d["allow"] else ("STEP_UP" if d["step_up"] else "DENY")
    return kind, ", ".join(d["reasons"]) or "-"


DEV = {
    "관리·패치됨": {"managed": True, "owner": None, "edr": True, "patched": True},
    "관리·패치 안 됨": {"managed": True, "owner": None, "edr": True, "patched": False},
    "EDR 꺼짐": {"managed": True, "owner": None, "edr": False, "patched": True},
    "개인 PC": {"managed": False},
}
rows = []
for user, role in [("kim", "employee"), ("lee", "partner")]:
    for app in ["erp", "aichat"]:
        for dev_name, dev in DEV.items():
            for method in ["GET", "POST"]:
                device = {**dev, "owner": user} if dev.get("managed") else dev
                rows.append((f"{user}({role})", app, dev_name, method,
                             *opa_decide(user=user, roles=[role], app=app, method=method, device=device, country="KR")))
table(["사용자", "앱", "기기", "메서드", "결정", "사유"], rows)

> 🙋 (생각해 보기)
> 위 표에서 ALLOW는 몇 칸인가요? 제로트러스트 정책은 "허용할 조건을 모두 갖춘 경우"만 열어 주므로, 표의 대부분이 DENY로 채워집니다.

## 3. 신원과 역할에 따라 달라지는 결과

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 랩 구성 | ✅ 완료 |
| 1 | 실습 파일 준비와 실행 | ✅ 완료 |
| 2 | 정책 엔진의 규칙 읽기 | ✅ 완료 |
| 3 | 신원과 역할에 따라 달라지는 결과 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | 신원은 정상인데 차단되는 경우 | 예정 |
| 5 | 우회해 보기 | 예정 |
| 6 | 위험 탐지가 정책을 바꾸는 순간 | 예정 |
| 7 | Trace ID로 한 요청 따라가기 | 예정 |
| 8 | 체크포인트와 과제 | 예정 |
| 9 | 정리 | 예정 |

이제 실제로 토큰을 받아 PEP를 거쳐 요청합니다. README의 `./try.sh kim LAPTOP-KIM-01 erp`와 같습니다.

In [ ]:
%%bash
cd ~/zt-lab && ./try.sh kim LAPTOP-KIM-01 erp

In [ ]:
show([
    ("kim", "LAPTOP-KIM-01", "erp"),        # 임직원 → ERP
    ("lee", "LAPTOP-LEE-01", "erp"),        # 협력사 → ERP
    ("lee", "LAPTOP-LEE-01", "aichat"),     # 협력사 → AI 챗봇
])

같은 IdP, 같은 게이트웨이를 쓰지만 역할에 따라 닿을 수 있는 시스템이 다릅니다. ERP와 AI 챗봇이 서로 다른 보안 체계를 갖지 않고 같은 네 계층을 공유한다는 점이 핵심입니다.

## 4. 신원은 정상인데 차단되는 경우

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 랩 구성 | ✅ 완료 |
| 1 | 실습 파일 준비와 실행 | ✅ 완료 |
| 2 | 정책 엔진의 규칙 읽기 | ✅ 완료 |
| 3 | 신원과 역할에 따라 달라지는 결과 | ✅ 완료 |
| 4 | 신원은 정상인데 차단되는 경우 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | 우회해 보기 | 예정 |
| 6 | 위험 탐지가 정책을 바꾸는 순간 | 예정 |
| 7 | Trace ID로 한 요청 따라가기 | 예정 |
| 8 | 체크포인트와 과제 | 예정 |
| 9 | 정리 | 예정 |

In [ ]:
show([
    ("kim", "MY-HOME-PC", "erp"),                     # 등록되지 않은 개인 PC
    ("lee", "LAPTOP-LEE-01", "aichat", "KR", "POST"), # 패치 안 된 기기로 쓰기 요청
])

kim의 비밀번호와 토큰은 정상이지만 요청은 막혔습니다. 제로트러스트의 판단 근거는 "누구인가" 하나가 아닙니다.

### 4.1. 기기를 패치하면

`devices.json`에서 `LAPTOP-LEE-01`의 `"patched"`를 `true`로 바꾸고 PEP를 다시 시작한 뒤 다시 보냅니다. 실제 환경에서는 이 기기 정보를 MDM·EDR이 실시간으로 알려 줍니다.

> 📌 (참고) `devices.json`은 PEP 컨테이너에 파일 단위로 마운트되어 있으므로, 파일을 새로 만들지 않고 같은 파일에 덮어씁니다.

In [ ]:
import time

devices_file = WORK / "devices.json"
devices = json.loads(devices_file.read_text())
devices["LAPTOP-LEE-01"]["patched"] = True
devices_file.write_text(json.dumps(devices, indent=2))
print(sh("cd ~/zt-lab && docker compose restart pep 2>&1 | tail -1"))
time.sleep(3)
show([("lee", "LAPTOP-LEE-01", "aichat", "KR", "POST")])

## 5. 우회해 보기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 랩 구성 | ✅ 완료 |
| 1 | 실습 파일 준비와 실행 | ✅ 완료 |
| 2 | 정책 엔진의 규칙 읽기 | ✅ 완료 |
| 3 | 신원과 역할에 따라 달라지는 결과 | ✅ 완료 |
| 4 | 신원은 정상인데 차단되는 경우 | ✅ 완료 |
| 5 | 우회해 보기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 위험 탐지가 정책을 바꾸는 순간 | 예정 |
| 7 | Trace ID로 한 요청 따라가기 | 예정 |
| 8 | 체크포인트와 과제 | 예정 |
| 9 | 정리 | 예정 |

PEP를 건너뛰고 앱에 직접 붙어 봅니다.

In [ ]:
%%bash
cd ~/zt-lab
echo "── 내 PC에서 ERP로 직접"
curl -s -m 3 http://localhost:9000 || echo "연결 실패(우회 불가)"
echo "── PEP 컨테이너 안에서 내부망의 ERP로 직접"
docker compose exec -T pep python -c "import httpx;print(httpx.get('http://erp:9000/home').json())"

첫 번째는 연결되지 않습니다. 앱이 포트를 열지 않았기 때문입니다. 두 번째는 PEP 컨테이너 안에서 내부망으로 직접 호출한 것이라 응답이 옵니다. **토큰도 정책 판단도 없이** ERP가 답했습니다.

이 결과가 뜻하는 바를 생각해 보세요. 집행 지점 자체가 침해되면 뒤의 모든 것이 열립니다. 그래서 PEP는 가장 단단하게 지켜야 하는 구성요소이고, 앱도 `X-Verified-User` 헤더만 믿지 말고 PEP와의 통신을 상호 인증(mTLS)으로 확인해야 합니다.

같은 내부망의 다른 컨테이너는 어떨까요? `detector`도 `back` 내부망에 있습니다.

In [ ]:
%%bash
cd ~/zt-lab
docker compose exec -T detector python -c "import httpx;print(httpx.get('http://erp:9000/home', headers={'X-Verified-User':'ceo'}).json())"

위험 탐지기 컨테이너에서도 ERP에 닿고, `X-Verified-User` 헤더를 **아무 이름(`ceo`)으로 꾸며도** 앱은 그대로 믿습니다. 내부망에 있다는 것만으로 신뢰하는 구조가 남아 있다는 뜻입니다. 7-3의 마이크로 세그멘테이션(앱마다 PEP에서 오는 연결만 허용)이 이 틈을 막습니다.

## 6. 위험 탐지가 정책을 바꾸는 순간

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 랩 구성 | ✅ 완료 |
| 1 | 실습 파일 준비와 실행 | ✅ 완료 |
| 2 | 정책 엔진의 규칙 읽기 | ✅ 완료 |
| 3 | 신원과 역할에 따라 달라지는 결과 | ✅ 완료 |
| 4 | 신원은 정상인데 차단되는 경우 | ✅ 완료 |
| 5 | 우회해 보기 | ✅ 완료 |
| 6 | 위험 탐지가 정책을 바꾸는 순간 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 7 | Trace ID로 한 요청 따라가기 | 예정 |
| 8 | 체크포인트와 과제 | 예정 |
| 9 | 정리 | 예정 |

이제 kim이 몇 초 사이에 미국에서 접속한 것처럼 요청을 보냅니다. 비밀번호가 유출되어 다른 나라의 공격자가 로그인한 상황을 흉내 낸 것입니다.

In [ ]:
show([("kim", "LAPTOP-KIM-01", "erp", "US")])     # 미국에서 접속

미국 접속 요청 자체는 허용되었습니다. 그 순간에는 판단할 근거가 없었기 때문입니다. ④ 위험 탐지기는 3초마다 로그를 읽습니다. 몇 초 기다린 뒤 탐지기 기록과 정책 엔진이 보고 있는 위험 점수를 확인합니다.

In [ ]:
%%bash
sleep 5
cd ~/zt-lab
docker compose logs detector | tail -3
echo "── 정책 엔진의 위험 점수"
curl -s localhost:8181/v1/data/risk

In [ ]:
show([("kim", "LAPTOP-KIM-01", "erp")])            # 다시 한국에서 접속

④ 위험 탐지기가 로그에서 "5분 안에 두 나라"라는 신호를 찾아 위험 점수를 올렸고, 정책 엔진은 같은 사용자의 다음 요청에 추가 인증을 요구했습니다. 사람이 개입하지 않고 탐지에서 정책 변경까지 몇 초 만에 이어졌습니다. 본문에서 말한 "탐지 결과가 보고서로 끝나지 않고 정책으로 되돌아가는 것"이 이것입니다.

### 6.1. 거부가 쌓이면 차단으로

여기서 몇 번 더 거부되면 점수가 70을 넘어 STEP_UP이 DENY로 바뀝니다. 등록되지 않은 기기로 거듭 시도하는 공격자를 흉내 냅니다.

In [ ]:
for i in range(3):
    try_("kim", "UNKNOWN-PC", "erp")      # 거부 3번
time.sleep(5)
print(sh("curl -s localhost:8181/v1/data/risk"))
show([("kim", "LAPTOP-KIM-01", "erp")])

정상 기기로 돌아와도 이제는 차단됩니다. 위험 점수는 최근 5분의 로그만 보므로, 5분이 지나면 저절로 내려갑니다.

> 🙋 (생각해 보기)
> 공격자가 일부러 거부를 일으켜 **정상 사용자 kim을 잠기게** 만들 수도 있을까요? 탐지 결과를 자동으로 정책에 반영할 때는, 그 자동화가 서비스 거부(DoS) 수단이 되지 않는지도 함께 따져야 합니다.

## 7. Trace ID로 한 요청 따라가기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 랩 구성 | ✅ 완료 |
| 1 | 실습 파일 준비와 실행 | ✅ 완료 |
| 2 | 정책 엔진의 규칙 읽기 | ✅ 완료 |
| 3 | 신원과 역할에 따라 달라지는 결과 | ✅ 완료 |
| 4 | 신원은 정상인데 차단되는 경우 | ✅ 완료 |
| 5 | 우회해 보기 | ✅ 완료 |
| 6 | 위험 탐지가 정책을 바꾸는 순간 | ✅ 완료 |
| 7 | Trace ID로 한 요청 따라가기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 8 | 체크포인트와 과제 | 예정 |
| 9 | 정리 | 예정 |

모든 응답에는 `trace`가 붙어 있고, 같은 값이 `logs/access.jsonl`에 남습니다. 6단계의 마지막 요청을 따라가 봅니다.

In [ ]:
last = try_("kim", "LAPTOP-KIM-01", "erp")
print("응답:", last)
print("── 로그에서 같은 Trace ID 찾기")
print(sh(f"grep {last['trace']} ~/zt-lab/logs/access.jsonl"))

누가, 어떤 기기로, 어디서, 무엇을 요청했고, 왜 그런 결정이 내려졌는지가 한 줄에 있습니다. 9-2의 Trace ID 기반 추적이 이 기록을 사용자 요청부터 모델 호출, Tool 실행까지 넓힌 것입니다.

### 7.1. 접근 로그 전체 집계

지금까지의 모든 요청을 사용자·결정별로 집계합니다. 관제 화면이 보여 주는 숫자가 이런 집계입니다.

In [ ]:
from collections import Counter

events = [json.loads(l) for l in (WORK / "logs" / "access.jsonl").read_text().splitlines()]
c = Counter((e["user"], e["app"], e["decision"]) for e in events)
table(["사용자", "앱", "결정", "횟수"], [(*k, n) for k, n in sorted(c.items(), key=str)])

## 8. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 랩 구성 | ✅ 완료 |
| 1 | 실습 파일 준비와 실행 | ✅ 완료 |
| 2 | 정책 엔진의 규칙 읽기 | ✅ 완료 |
| 3 | 신원과 역할에 따라 달라지는 결과 | ✅ 완료 |
| 4 | 신원은 정상인데 차단되는 경우 | ✅ 완료 |
| 5 | 우회해 보기 | ✅ 완료 |
| 6 | 위험 탐지가 정책을 바꾸는 순간 | ✅ 완료 |
| 7 | Trace ID로 한 요청 따라가기 | ✅ 완료 |
| 8 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 9 | 정리 | 예정 |

### 8.1. 체크포인트

- [ ] 신원이 정상인데 차단된 요청은 어떤 근거 때문이었나요? (기기, 역할, 패치 상태, 위험 점수)
- [ ] 미국 접속 요청이 처음에는 허용된 이유는 무엇인가요? 탐지와 판단 사이의 몇 초를 더 줄이려면 무엇이 필요할까요?
- [ ] 이 랩에서 ① 사용자 계층의 기기 정보(`devices.json`)는 실제로는 어떤 시스템에서 받아야 할까요?
- [ ] PEP 컨테이너가 침해되면 어떤 일이 생기나요? 그림 7-2의 어느 계층이 이를 알아챌 수 있을까요?

### 8.2. 과제 1: 정책 규칙과 테스트 추가하기

6단계에서 미국 접속은 처음 한 번은 허용되었습니다. 회사가 "ERP는 국내(KR)에서만 접속"하기로 정했다면, 탐지를 기다리지 않고 정책에서 바로 막을 수 있습니다.

1. 아래 셀의 `NEW_RULE`에 "ERP를 KR이 아닌 국가에서 요청하면 차단" 규칙을 씁니다.
2. `NEW_TEST`에 이 규칙을 확인하는 테스트를 씁니다.
3. 셀을 실행하면 두 파일 끝에 덧붙이고, `opa test`를 돌린 뒤 OPA를 다시 시작해 실제 요청으로 확인합니다.

> 📌 (참고) OPA는 시작할 때 정책을 읽으므로 고친 뒤 다시 시작합니다. 다시 시작하면 메모리에 있던 위험 점수도 지워집니다.

In [ ]:
NEW_RULE = """
# TODO: ERP를 KR이 아닌 국가에서 요청하면 차단
"""

NEW_TEST = """
# TODO: kim_ok에서 country만 US로 바꾸면 차단되어야 한다
"""

for name, extra in [("zt.rego", NEW_RULE), ("zt_test.rego", NEW_TEST)]:
    f = WORK / "policy" / name
    f.write_text(f.read_text() + extra)
print(sh("cd ~/zt-lab && docker compose exec -T opa opa test /policy -v 2>&1"))
print(sh("cd ~/zt-lab && docker compose restart opa 2>&1 | tail -1"))
time.sleep(3)
show([("kim", "LAPTOP-KIM-01", "erp", "US"), ("kim", "LAPTOP-KIM-01", "aichat", "US"), ("kim", "LAPTOP-KIM-01", "erp")])

<details>
<summary>💡 힌트 보기</summary>

```python
NEW_RULE = """
deny contains "허용되지 않은 국가에서 ERP 접속" if {
	input.app == "erp"
	input.country != "KR"
}
"""

NEW_TEST = """
test_erp_from_us_denied if {
	d := zt.decision with input as object.union(kim_ok, {"country": "US"})
	"허용되지 않은 국가에서 ERP 접속" in d.reasons
}
"""
```

`kim_ok`에는 `country`가 없으므로, 규칙을 `input.country != "KR"`로 쓰면 국가 정보가 **없는** 요청은 이 규칙에 걸리지 않습니다(Rego에서 없는 값과의 비교는 성립하지 않음). 기존 테스트 5개가 그대로 통과하는 이유이기도 합니다. 운영 정책이라면 "국가 정보가 없으면 차단"까지 함께 정해야 합니다. PEP는 헤더가 없으면 `KR`로 채우므로, 이 랩에서는 클라이언트가 국가를 속일 수 있다는 점도 함께 생각해 보세요.
</details>

### 8.3. 과제 2: 이 랩의 빈틈 정리하기 (서술)

5단계와 6단계에서 찾은 빈틈을 정리하고, 7장 본문의 어떤 통제로 막을지 적어 보세요.

| 빈틈 | 실습에서 확인한 내용 | 막는 통제 (7장의 절) |
|---|---|---|
| PEP 침해 시 내부망 전체 노출 | | |
| 내부망 컨테이너가 사용자 헤더를 꾸밈 | | |
| 접속 국가를 클라이언트가 헤더로 보냄 | | |
| 거부 유발로 정상 사용자 잠금 | | |

## 9. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 랩 구성 | ✅ 완료 |
| 1 | 실습 파일 준비와 실행 | ✅ 완료 |
| 2 | 정책 엔진의 규칙 읽기 | ✅ 완료 |
| 3 | 신원과 역할에 따라 달라지는 결과 | ✅ 완료 |
| 4 | 신원은 정상인데 차단되는 경우 | ✅ 완료 |
| 5 | 우회해 보기 | ✅ 완료 |
| 6 | 위험 탐지가 정책을 바꾸는 순간 | ✅ 완료 |
| 7 | Trace ID로 한 요청 따라가기 | ✅ 완료 |
| 8 | 체크포인트와 과제 | ✅ 완료 |
| 9 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

In [ ]:
%%bash
cd ~/zt-lab && docker compose down -v -t 1 2>&1 | grep -cE 'Removed' | xargs echo "삭제한 항목 수:"
cd ~ && rm -rf ~/zt-lab && echo "실습 폴더 삭제됨"

---
> 다음 실습 3에서는 같은 역할(employee, partner) 정보가 RAG 검색 단계에서 문서 권한을 거르는 데 어떻게 쓰이는지 확인합니다.